# CairoGo Team Analysis — Required Evidence Notebook

Rename to `team_analysis.ipynb` and place at the **root of `T_<leaderID>.zip`**.

- Not an extra graded component; a reproducible record of Parts 1–2 for spot-audits and discussion.
- Run it with **your assigned CairoGo data pack**. You may import included `.py` modules.
- Describe your split, preprocessing, algorithms, evaluation, and decisions. Keep complete derivations.
- Do not embed entire raw datasets, private secrets, or large tables in outputs.
- Your final `results.json`, `predict.py`, `policy.py`, and CSVs remain authoritative outputs.

Replace TODO cells with **your actual code and evidence**; the scaffold contains no answers.

## 1. Data audit & Join

Show zone name matching before and after fixing legacy names; calculate unmatched share before repair, and demonstrate the shift, leakage and confounding checks.

In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import roc_auc_score
import preprocess

DATA_DIR = "team_data/CairoGo_pack_v03"
trips = pd.read_csv(f"{DATA_DIR}/trips.csv")
recent = pd.read_csv(f"{DATA_DIR}/recent_labelled.csv")
zones = pd.read_csv(f"{DATA_DIR}/zones.csv")

# 1a. Join loss
valid_zones = set(zones["zone_name"].dropna())
unmatched_mask = ~trips["pickup_zone"].isin(valid_zones)
unmatched_before = float(unmatched_mask.mean())
unmatched_zones = sorted(trips.loc[unmatched_mask, "pickup_zone"].dropna().unique().tolist())
print(f"1a. Unmatched pickup_zone share: {unmatched_before:.6f}, zones: {unmatched_zones}")

trips_fixed_zones = preprocess.clean_requests(trips, fix_zones=True, fix_unit_shift=False, drop_leakage=False)
print("    Unmatched after fix:", int((~trips_fixed_zones["pickup_zone"].isin(valid_zones)).sum()))

# 1b. Unit shift (trips vs recent)
print("1b. Median distance_km before 2026-08-27:", round(float(trips["distance_km"].median()), 2),
      "| from 2026-08-27:", round(float(recent["distance_km"].median()), 2))

# 1c. Leakage
y = trips["request_failed"].values
for col in preprocess.LEAKAGE_COLUMNS:
    auc_nn = roc_auc_score(y, trips[col].notna().astype(int))
    print(f"1c. Leakage col {col}: not-null AUC = {auc_nn:.4f}")

# 1d. Confounding
comp = trips_fixed_zones.dropna(subset=["actual_trip_time_min"]).merge(
    zones, left_on="pickup_zone", right_on="zone_name", how="left"
)
global_slope = float(LinearRegression().fit(comp[["distance_km"]], comp["actual_trip_time_min"]).coef_[0])
weights = []
for zt, df_g in comp.groupby("zone_type"):
    s = float(LinearRegression().fit(df_g[["distance_km"]], df_g["actual_trip_time_min"]).coef_[0])
    weights.append((s, len(df_g)))
within_slope = sum(s * w for s, w in weights) / sum(w for _, w in weights)
print(f"1d. Confounder=zone_type | global_slope={global_slope:.4f} | within_slope={within_slope:.4f}")

## 2. Missingness & preprocessing

Document imputation/missingness features, leakage prevention and how you fit transformations only on earlier training data.

In [ ]:
drivers = pd.read_csv(f"{DATA_DIR}/drivers.csv")
customers = pd.read_csv(f"{DATA_DIR}/customers.csv")

# Task 2: Missingness (MNAR)
w_miss = trips["weather"].isna()
w_diff = float(trips.loc[w_miss, "request_failed"].mean() - trips.loc[~w_miss, "request_failed"].mean())

td = trips.merge(drivers[["driver_id", "rating_avg"]], on="driver_id", how="left")
r_miss = td["rating_avg"].isna()
r_diff = float(td.loc[r_miss, "request_failed"].mean() - td.loc[~r_miss, "request_failed"].mean())
print(f"2. Weather missing={w_miss.mean():.4f} (fail_diff={w_diff:+.4f}) | Rating missing={r_miss.mean():.4f} (fail_diff={r_diff:+.4f})")

# Shared preprocessing & time-based split
clean_df = preprocess.build_features(trips, zones_df=zones, customers_df=customers, drivers_df=drivers)
train_df, val_df = preprocess.time_split(clean_df, val_frac=0.2)
print(f"Clean shape={clean_df.shape} | train={len(train_df)}, val={len(val_df)}")

## 3. Model comparison

Same chronological split for Logistic Regression, boosted tree, kNN and MLP; show AUC and Recall@15; compute Brier/cost as applicable and explain any differences in results.json field availability.

In [ ]:
# TODO: insert your reproducible experiment here. You may import a local .py module.
# Example placeholder only — replace before submission:
pass

## 3b. Predictions & calibration

Show final model selection, probability predictions and your VALIDATION-BASED holdout recall estimate; hidden true holdout recall is unknown to you.

In [ ]:
# TODO: insert your reproducible experiment here. You may import a local .py module.
# Example placeholder only — replace before submission:
pass

## 4. Customer segmentation

Show customer profile construction, scaling, K-means fitting, silhouette, stability and interpretations.

In [ ]:
# TODO: insert your reproducible experiment here. You may import a local .py module.
# Example placeholder only — replace before submission:
pass

## 5. Local-k

Show your from-scratch Local-k implementation or import its .py file; five forward-in-time comparisons, cost deltas and failure analyses.

In [ ]:
# TODO: insert your reproducible experiment here. You may import a local .py module.
# Example placeholder only — replace before submission:
pass

## 6. Recent-data investigation

Show unit/policy shift detection, repair and validation of its performance effect; ensure repair is also in predict.py.

In [ ]:
# TODO: insert your reproducible experiment here. You may import a local .py module.
# Example placeholder only — replace before submission:
pass

## 7. Text reviews

Show actual NB and SVM pipelines and identify five genuine disagreements; explain each; no topic-modelling task.

In [ ]:
# TODO: insert your reproducible experiment here. You may import a local .py module.
# Example placeholder only — replace before submission:
pass

## 8. Decision analysis

Show estimation under published rules, baselines, customer segments vs no segments, calibration, uncertainty and group errors; no hidden labels.

In [ ]:
# TODO: insert your reproducible experiment here. You may import a local .py module.
# Example placeholder only — replace before submission:
pass